In [ ]:
from pathlib import Path
import pandas as pd
import os

folder_path = 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\12'  # e.g., 'C:/Users/Documents' or './data'
file_name = 'market_orders.csv'

file_path = os.path.join(folder_path, file_name)

print(f"Ready to load file from: {file_path}")

In [ ]:
# 1. Define the full list of headers you provided
cols_to_keep = [
    'start_time', 'accepted_price', 'accepted_volume', 'bid_id', 
    'bid_type', 'end_time', 'market_id', 'max_power', 
    'min_power', 'node', 'price', 'simulation', 'unit_id', 'volume'
]

# 2. Define your target units
target_unit_ids = [
    "P100000124212", "P100000124839", "P100000125777", "P100000125790",
    "P100000125763", "P100000125764", "P100000125776", "P100000125770"
]

filtered_pieces = []
chunk_size = 500000 

print("Processing full dataset... this will take a moment.")

# We still use chunks to prevent the 6GB file from crashing your RAM
for chunk in pd.read_csv(file_path, usecols=cols_to_keep, chunksize=chunk_size):
    # Apply your filters
    mask = (chunk['market_id'] == 'CRM_neg') & (chunk['unit_id'].isin(target_unit_ids))
    
    # Keep the matching rows
    filtered_pieces.append(chunk[mask])

# Combine all found rows into one DataFrame
if filtered_pieces:
    final_df = pd.concat(filtered_pieces)
    print(f"Done! Found {len(final_df)} rows with all {len(final_df.columns)} columns.")
    display(final_df.head())
else:
    print("No matches found. Please check if 'market_id' or 'unit_id' values match exactly.")

In [ ]:
# Calculate the grand totals for the filtered data
total_volume = final_df['volume'].sum()
total_accepted_volume = final_df['accepted_volume'].sum()

print(f"Total Volume: {total_volume:,.2f}")
print(f"Total Accepted Volume: {total_accepted_volume:,.2f}")

# Optional: If you want to see the breakdown for each specific Unit ID
print("\n--- Breakdown by Unit ID ---")
summary_by_unit = final_df.groupby('unit_id')[['volume', 'accepted_volume']].sum()
display(summary_by_unit)

In [ ]:
# Group by unit_id and calculate min, max, and mean (average) for price and accepted_price
price_stats = final_df.groupby('unit_id').agg({
    'price': ['min', 'max', 'mean'],
    'accepted_price': ['min', 'max', 'mean']
})

# Flatten the column names to make it easier to read
price_stats.columns = [
    'Price_Min', 'Price_Max', 'Price_Avg', 
    'Acc_Price_Min', 'Acc_Price_Max', 'Acc_Price_Avg'
]

# Display the results
print("--- Price Statistics by Unit ID ---")
display(price_stats)

Aggregated Analysis for CRM_pos

In [ ]:
# List your scenario file names or folder paths
# Update these names to match your actual files exactly
scenarios = {
    'SEFES35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\10\\market_orders.csv',
    'SEPHS35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\11\\market_orders.csv',
    'SEPES35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\12\\market_orders.csv'
}

target_unit_ids = [
    "P100000124212", "P100000124839", "P100000125777", "P100000125790",
    "P100000125763", "P100000125764", "P100000125776", "P100000125770"
]

cols_to_keep = [
    'start_time', 'accepted_price', 'accepted_volume', 'bid_id', 
    'bid_type', 'end_time', 'market_id', 'max_power', 
    'min_power', 'node', 'price', 'simulation', 'unit_id', 'volume'
]

In [ ]:
import pandas as pd
all_filtered_data = []

for name, path in scenarios.items():
    print(f"Processing {name}...")
    
    # Process in chunks to save RAM
    for chunk in pd.read_csv(path, usecols=cols_to_keep, chunksize=500000):
        # Filter for market and units
        mask = (chunk['market_id'] == 'CRM_pos') & (chunk['unit_id'].isin(target_unit_ids))
        
        # Add a column to identify which scenario this row belongs to (if not already in 'simulation')
        filtered_chunk = chunk[mask].copy()
        filtered_chunk['scenario_label'] = name 
        
        all_filtered_data.append(filtered_chunk)

# Combine everything into one master table
master_df = pd.concat(all_filtered_data, ignore_index=True)

print(f"Success! Master DataFrame created with {len(master_df)} total rows across all scenarios.")
display(master_df.head())

In [ ]:
# Count rows per scenario
print(master_df['scenario_label'].value_counts())

In [ ]:
# Group by Scenario to compare performance
scenario_comparison = master_df.groupby('scenario_label').agg({
    'volume': ['sum', 'mean'],
    'accepted_volume': ['sum', 'mean'],
    'price': ['mean', 'max'],
    'accepted_price': ['mean', 'max']
})

# Clean up column names for readability
scenario_comparison.columns = [
    'Total_Vol', 'Avg_Vol', 
    'Total_Acc_Vol', 'Avg_Acc_Vol',
    'Avg_Price', 'Max_Price',
    'Avg_Acc_Price', 'Max_Acc_Price'
]

print("--- Multi-Scenario Performance Summary ---")
display(scenario_comparison)

In [ ]:
# Save the scenario comparison summary to a CSV file
# We keep index=True so that the Scenario labels (Scenario 10, 11, 12) are included
scenario_comparison.to_csv('scenario_performance_summary.csv', index=True)

print("The file 'scenario_performance_summary.csv' has been created in your folder.")

In [ ]:
import matplotlib.pyplot as plt

# Prepare data for plotting
plot_data = master_df.groupby('scenario_label')[['volume', 'accepted_volume']].sum()

# Create the plot
ax = plot_data.plot(kind='bar', figsize=(10, 6), color=['#3498db', '#2ecc71'])

plt.title('Comparison of Volume vs. Accepted Volume across Scenarios', fontsize=14)
plt.ylabel('Volume Total (TW)')
plt.xlabel('Scenario')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(["Total Bid Volume", "Total Accepted Volume"])

plt.show()

In [ ]:
# 1. Calculate the Success Rate (%) for each unit in each scenario
success_analysis = master_df.groupby(['scenario_label', 'unit_id']).agg({
    'volume': 'sum',
    'accepted_volume': 'sum'
}).reset_index()

success_analysis['success_rate_pct'] = (success_analysis['accepted_volume'] / success_analysis['volume']) * 100

# 2. Pivot the data to make it easy to compare scenarios side-by-side
success_pivot = success_analysis.pivot(index='unit_id', columns='scenario_label', values='success_rate_pct')

print("--- Unit Success Rate (%) Comparison ---")
display(success_pivot.round(2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(
    x='scenario_label',
    y='price',
    data=master_df,
    palette='Set2'
)

plt.title('aFRR (pos): Price Distribution and Volatility per Scenario')
plt.ylabel('EUR/MW')   # <-- set y-axis label
plt.xlabel('Scenario') # optional, for clarity

plt.show()


In [ ]:
# 1. Calculate Revenue for every row
master_df['revenue'] = master_df['accepted_volume'] * master_df['accepted_price']

# 2. Group by scenario and unit to get total revenue
revenue_analysis = master_df.groupby(['scenario_label', 'unit_id'])['revenue'].sum().reset_index()

# 3. Pivot for easy side-by-side comparison
revenue_pivot = revenue_analysis.pivot(index='unit_id', columns='scenario_label', values='revenue')

print("--- Total Revenue per Unit by Scenario ---")
# Display with formatting for currency
display(revenue_pivot.style.format("{:,.2f}"))

# 4. Save to CSV
# revenue_pivot.to_csv('unit_revenue_comparison.csv')
# print("\nFile 'unit_revenue_comparison.csv' has been saved to your folder.")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Prepare the data: Average Price and Success Rate per unit per scenario
efficiency_data = master_df.groupby(['scenario_label', 'unit_id']).agg({
    'price': 'mean',
    'volume': 'sum',
    'accepted_volume': 'sum'
}).reset_index()

efficiency_data['success_rate_pct'] = (efficiency_data['accepted_volume'] / efficiency_data['volume']) * 100

# 2. Create the Scatter Plot
plt.figure(figsize=(11, 7))
sns.scatterplot(
    data=efficiency_data, 
    x='price', 
    y='success_rate_pct', 
    hue='scenario_label', 
    style='scenario_label', 
    s=150,
    edgecolor='black',
    alpha=0.8
)

# 3. Annotate the dots with Unit IDs so you know which is which
for i in range(efficiency_data.shape[0]):
    plt.text(
        x=efficiency_data.price[i] + 0.5, 
        y=efficiency_data.success_rate_pct[i], 
        s=efficiency_data.unit_id[i], 
        fontsize=9, 
        alpha=0.8
    )

plt.title('Efficiency Mapping: How aFRR (pos) Price Affects Success Rate', fontsize=15)
plt.xlabel('Average Bid Price', fontsize=12)
plt.ylabel('Success Rate (%)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(title='Scenario', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.show()

In [ ]:
# 1. Convert end_time to actual datetime objects first
master_df['end_time'] = pd.to_datetime(master_df['end_time'])

# 2. Now extract the hour and create the heatmap data
master_df['hour'] = master_df['end_time'].dt.hour

heatmap_data = master_df.pivot_table(
    index='hour', 
    columns='scenario_label', 
    values='accepted_price', 
    aggfunc='mean'
)

# 3. Plotting the Heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="YlOrRd")

plt.title('Hourly Price Heatmap for the aFRR (pos): When was the market most expensive?')
plt.xlabel('Scenario')
plt.ylabel('Hour of Day (24h format)')
plt.show()

Aggregated Analysis for CRM_neg

In [ ]:
all_filtered_neg = []

# We use the same scenarios and paths defined earlier
for name, path in scenarios.items():
    print(f"Extracting CRM_neg from {name}...")
    
    for chunk in pd.read_csv(path, usecols=cols_to_keep, chunksize=500000):
        # Change the filter to CRM_neg
        mask = (chunk['market_id'] == 'CRM_neg') & (chunk['unit_id'].isin(target_unit_ids))
        
        filtered_chunk = chunk[mask].copy()
        filtered_chunk['scenario_label'] = name 
        all_filtered_neg.append(filtered_chunk)

master_df_neg = pd.concat(all_filtered_neg, ignore_index=True)
print(f"Done! Found {len(master_df_neg)} rows for CRM_neg.")

In [ ]:
# Create the Master CRM_neg DataFrame
master_df_neg = pd.concat(all_filtered_neg, ignore_index=True)

# Generate Scenario Comparison Summary for CRM_neg
neg_scenario_summary = master_df_neg.groupby('scenario_label').agg({
    'volume': ['sum', 'mean'],
    'accepted_volume': ['sum', 'mean'],
    'price': ['mean', 'max'],
    'accepted_price': ['mean', 'max']
})

neg_scenario_summary.columns = [
    'Total_Vol', 'Avg_Vol', 'Total_Acc_Vol', 'Avg_Acc_Vol',
    'Avg_Price', 'Max_Price', 'Avg_Acc_Price', 'Max_Acc_Price'
]

# Save and Display
neg_scenario_summary.to_csv('crm_neg_scenario_summary.csv', index=True)
print("\n--- CRM_neg Multi-Scenario Summary ---")
display(neg_scenario_summary.round(2))

In [ ]:
# 1. Calculate Revenue and Success Rate
master_df_neg['revenue'] = master_df_neg['accepted_volume'] * master_df_neg['accepted_price']

neg_unit_analysis = master_df_neg.groupby(['scenario_label', 'unit_id']).agg({
    'volume': 'sum',
    'accepted_volume': 'sum',
    'revenue': 'sum'
}).reset_index()

neg_unit_analysis['success_rate_pct'] = (neg_unit_analysis['accepted_volume'] / neg_unit_analysis['volume']) * 100

# 2. Create Pivot Tables for CSV Export
neg_revenue_pivot = neg_unit_analysis.pivot(index='unit_id', columns='scenario_label', values='revenue')
neg_success_pivot = neg_unit_analysis.pivot(index='unit_id', columns='scenario_label', values='success_rate_pct')

# Save Files
neg_revenue_pivot.to_csv('crm_neg_revenue_comparison.csv')
neg_success_pivot.to_csv('crm_neg_success_rate_comparison.csv')

print("CRM_neg Revenue and Success Rate files saved.")
display(neg_success_pivot.round(2))

In [ ]:
# Convert to datetime and extract hour
master_df_neg['end_time'] = pd.to_datetime(master_df_neg['end_time'])
master_df_neg['hour'] = master_df_neg['end_time'].dt.hour

# Pivot for heatmap
neg_heatmap_data = master_df_neg.pivot_table(
    index='hour', 
    columns='scenario_label', 
    values='accepted_price', 
    aggfunc='mean'
)

# Plotting
plt.figure(figsize=(10, 8))
sns.heatmap(neg_heatmap_data, annot=True, fmt=".1f", cmap="Blues") # Using Blue for "Negative" market
plt.title('Hourly Price Heatmap for the aFRR (neg): When was the market most expensive?')
plt.xlabel('Scenario')
plt.ylabel('Hour of Day')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Price Distribution Visualization (Boxplot)
plt.figure(figsize=(10, 6))
sns.boxplot(
    x='scenario_label',
    y='price',
    data=master_df_neg,
    palette='vlag'
)

plt.title('aFRR (neg): Price Distribution and Volatility per Scenario', fontsize=14)
plt.ylabel('EUR/MW')   # <-- updated unit
plt.xlabel('Scenario')
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.show()


CRM Consolidated result

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Prepare Positive Data (from your master_df)
pos_vol = master_df.groupby('scenario_label')['accepted_volume'].sum()

# 2. Prepare Negative Data (from your master_df_neg)
neg_vol = master_df_neg.groupby('scenario_label')['accepted_volume'].sum()

# 3. Combine into a single comparison DataFrame
comparison_df = pd.DataFrame({
    'Positive_Accepted_Vol': pos_vol,
    'Negative_Accepted_Vol': neg_vol
})

# 4. Plotting
ax = comparison_df.plot(kind='bar', figsize=(12, 6), color=['#2ecc71', '#3498db'])

plt.title('Total Market Acceptance: CRM_pos vs. CRM_neg', fontsize=15)
plt.ylabel('Volume (MW)')
plt.xlabel('Scenario')
plt.xticks(rotation=0)
plt.legend(["Positive Reserve (Upward)", "Negative Reserve (Downward)"])
plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

# Export this final comparison table
comparison_df.to_csv('total_market_volume_comparison.csv')

In [ ]:
# Calculate total revenue per unit from both markets
pos_rev = master_df.groupby('unit_id')['revenue'].sum()
neg_rev = master_df_neg.groupby('unit_id')['revenue'].sum()

total_flex_revenue = pd.DataFrame({
    'POS_Revenue': pos_rev,
    'NEG_Revenue': neg_rev,
    'Total_Combined': pos_rev + neg_rev
}).sort_values(by='Total_Combined', ascending=False)

print("--- Top Units by Total Combined Revenue (Pos + Neg) ---")
display(total_flex_revenue.style.format("{:,.2f}")) # Ensure jinja2 is installed for this or use .round(2)

total_flex_revenue.to_csv('total_unit_revenue_combined.csv')

Day ahead analysis

In [ ]:
# Updated Scenario List (7 Total)
scenarios_eom = {
    'SE35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\2\\with_storage_trans\\market_orders.csv',
    'SEPE35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\4\\with_storage_trans\\market_orders.csv',
    'SEPH35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\5\\with_storage_trans\\market_orders.csv',
    'SEFE35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\6\\with_storage_trans\\market_orders.csv',
    'SEPES35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\10\\market_orders.csv',
    'SEPHS35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\11\\market_orders.csv',
    'SEFES35': 'C:\\Users\\khm\\ownCloud (2)\\Dropbox\\Ph.D\\My publications\\Journal paper\\3\\Simulation\\Output\\12\\market_orders.csv'
}

# The target_unit_ids and cols_to_keep remain the same as before

In [ ]:
all_filtered_eom = []

for name, path in scenarios_eom.items():
    print(f"Processing EOM for {name}...")
    for chunk in pd.read_csv(path, usecols=cols_to_keep, chunksize=500000):
        # Filter for EOM and your Cement Plant Units
        mask = (chunk['market_id'] == 'EOM') & (chunk['unit_id'].isin(target_unit_ids))
        
        filtered_chunk = chunk[mask].copy()
        filtered_chunk['scenario_label'] = name 
        all_filtered_eom.append(filtered_chunk)

master_df_eom = pd.concat(all_filtered_eom, ignore_index=True)
master_df_eom['end_time'] = pd.to_datetime(master_df_eom['end_time'])

print(f"Success! Captured {len(master_df_eom)} EOM records.")

In [ ]:
# 1. Update the Mapping
tech_map = {
    'SE35': 'BAU (Fossil)',
    'SEPE35': 'Electrified Only', 
    'SEPH35': 'Electrified Only', 
    'SEFE35': 'Electrified Only',
    'SEPES35': 'Electrified + Storage', 
    'SEPHS35': 'Electrified + Storage', 
    'SEFES35': 'Electrified + Storage'
}

master_df_eom['tech_group'] = master_df_eom['scenario_label'].map(tech_map)

# 2. Calculate Total Cost
master_df_eom['cost'] = master_df_eom['accepted_volume'] * master_df_eom['accepted_price']

# 3. Aggregate Metrics
eom_tech_summary = master_df_eom.groupby(['tech_group', 'scenario_label']).agg({
    'accepted_volume': 'sum',
    'cost': 'sum',
    'price': 'mean'
}).reset_index()

# 4. Calculate VWAP (Actual price paid per MWh)
eom_tech_summary['VWAP'] = eom_tech_summary['cost'] / eom_tech_summary['accepted_volume']

print("--- EOM Roadmap Analysis: BAU vs. Electrification vs. Storage ---")
display(eom_tech_summary.sort_values('VWAP').round(2))

In [ ]:
import matplotlib.pyplot as plt

# 1. Filter for the two comparison scenarios
target_scenarios = ['SEFE35', 'SEFES35']
comparison_df = master_df_eom[master_df_eom['scenario_label'].isin(target_scenarios)].copy()

# --- FIX START ---
# Ensure the timestamp is in datetime format and extract the hour
# Replace 'timestamp' with 'start_time' or 'date' if your column is named differently
if 'hour' not in comparison_df.columns:
    # Check if you have a timestamp column to extract from
    time_col = [c for c in ['timestamp', 'start_time', 'date'] if c in comparison_df.columns]
    if time_col:
        comparison_df['hour'] = pd.to_datetime(comparison_df[time_col[0]]).dt.hour
    else:
        # If the index is the timestamp, use this:
        comparison_df['hour'] = comparison_df.index.hour
# --- FIX END ---

# 2. Use absolute values for volume (Demand side)
comparison_df['abs_volume'] = comparison_df['accepted_volume'].abs()

# 3. Group by hour and scenario for Volume and Price
hourly_comparison = comparison_df.groupby(['hour', 'scenario_label']).agg({
    'abs_volume': 'sum',
    'accepted_price': 'mean'
}).reset_index()

# Pivot for easy plotting
volume_plot_data = hourly_comparison.pivot(index='hour', columns='scenario_label', values='abs_volume')
# Average price across these scenarios (usually identical per hour in these simulations)
price_trend = hourly_comparison.groupby('hour')['accepted_price'].mean()

# 4. Create the Plot
fig, ax1 = plt.subplots(figsize=(14, 7))

# Plotting Consumption (Primary Y-Axis)
volume_plot_data.plot(kind='line', ax=ax1, marker='o', linewidth=3, alpha=0.8)
ax1.set_ylabel('Total Electricity Consumption (MWh)', fontsize=12)
ax1.set_xlabel('Hour of Day (0-23)', fontsize=12)
ax1.set_title('E-TES Load Shifting in Response to Market Prices', fontsize=15)
ax1.legend(title='Scenario', loc='upper left')

# Plotting Market Price (Secondary Y-Axis)
ax2 = ax1.twinx()
ax2.plot(price_trend.index, price_trend.values, color='red', linestyle='--', linewidth=2, label='Market Price')
ax2.set_ylabel('Market Price (€/MWh)', color='red', fontsize=12)
ax2.tick_params(axis='y', labelcolor='red')

# Formatting
ax1.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# 5. Numerical Summary of Savings
savings_pct = ((27.12 - 19.23) / 27.12) * 100
print(f"Summary Insight:")
print(f" - SEFE35 (No Storage) VWAP: 27.12 €/MWh")
print(f" - SEFES35 (With Storage) VWAP: 19.23 €/MWh")
print(f" - Result: Thermal Storage achieved a {savings_pct:.1f}% reduction in average energy cost.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Define the pairs to compare
remaining_pairs = [
    ('SEPE35', 'SEPES35'), # Electrified vs. Electrified+Storage
    ('SEPH35', 'SEPHS35')  # Electrified vs. Electrified+Storage
]

# VWAP values from your previous summary table for calculation
vwap_data = {
    'SEPE35': 27.54, 'SEPES35': 22.14,
    'SEPH35': 26.64, 'SEPHS35': 31.26
}

for only_scen, storage_scen in remaining_pairs:
    # 1. Prepare Data
    target_scenarios = [only_scen, storage_scen]
    comparison_df = master_df_eom[master_df_eom['scenario_label'].isin(target_scenarios)].copy()

    # --- FIX START: Ensure 'hour' column exists ---
    if 'hour' not in comparison_df.columns:
        # Check for common timestamp column names
        time_col = [c for c in ['timestamp', 'start_time', 'date'] if c in comparison_df.columns]
        if time_col:
            comparison_df['hour'] = pd.to_datetime(comparison_df[time_col[0]]).dt.hour
        else:
            # Fallback to index if it's a DatetimeIndex
            try:
                comparison_df['hour'] = comparison_df.index.hour
            except AttributeError:
                print(f"Warning: Could not find time data for {only_scen}/{storage_scen}")
                continue
    # --- FIX END ---

    comparison_df['abs_volume'] = comparison_df['accepted_volume'].abs()

    # 2. Group by Hour
    hourly_data = comparison_df.groupby(['hour', 'scenario_label']).agg({
        'abs_volume': 'sum',
        'accepted_price': 'mean'
    }).reset_index()

    # Pivot for plotting
    v_plot = hourly_data.pivot(index='hour', columns='scenario_label', values='abs_volume')
    p_trend = hourly_data.groupby('hour')['accepted_price'].mean()

    # 3. Plotting
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Consumption Lines
    v_plot.plot(kind='line', ax=ax1, marker='o', linewidth=3, alpha=0.8)
    ax1.set_ylabel('Consumption (MWh)', fontsize=12)
    ax1.set_xlabel('Hour of Day (0-23)', fontsize=12)
    ax1.set_title(f'EOM Load Shifting: {only_scen} vs {storage_scen}', fontsize=14)
    ax1.legend(title='Scenario', loc='upper left')

    # Market Price Line
    ax2 = ax1.twinx()
    ax2.plot(p_trend.index, p_trend.values, color='red', linestyle='--', linewidth=2, label='Market Price')
    ax2.set_ylabel('Price (€/MWh)', color='red', fontsize=12)
    ax2.tick_params(axis='y', labelcolor='red')

    ax1.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

    # 4. Savings Calculation
    v_base = vwap_data[only_scen]
    v_store = vwap_data[storage_scen]
    savings = ((v_base - v_store) / v_base) * 100

    print(f"--- Analysis: {only_scen} vs {storage_scen} ---")
    print(f" - {only_scen} VWAP: {v_base} €/MWh")
    print(f" - {storage_scen} VWAP: {v_store} €/MWh")
    if savings > 0:
        print(f" - RESULT: Storage reduced energy costs by {savings:.1f}%.")
    else:
        print(f" - RESULT: Storage was {abs(savings):.1f}% MORE expensive. (Check for charging during peak hours)")
    print("-" * 50)

In [ ]:
import pandas as pd

# 1. Analyze EOM Data (Operational Cost)
# Using absolute volume because demand is negative in your model sign convention
master_df_eom['abs_vol'] = master_df_eom['accepted_volume'].abs()
master_df_eom['total_cost'] = master_df_eom['abs_vol'] * master_df_eom['accepted_price']

eom_stats = master_df_eom.groupby('scenario_label').agg({
    'abs_vol': 'sum',
    'total_cost': 'sum'
})
eom_stats['EOM_VWAP'] = eom_stats['total_cost'] / eom_stats['abs_vol']

# 2. Analyze CRM Data (Flexibility Performance)
# We calculate the actual success rate across all units in each scenario
crm_stats = master_df.groupby('scenario_label').agg({
    'volume': 'sum',
    'accepted_volume': 'sum'
})
crm_stats['CRM_Success_Rate_Pct'] = (crm_stats['accepted_volume'] / crm_stats['volume']) * 100

# 3. Merge Analyses into the Evidence Table
evidence_table = eom_stats[['EOM_VWAP']].merge(
    crm_stats[['CRM_Success_Rate_Pct']], 
    left_index=True, 
    right_index=True, 
    how='outer'
).reset_index()

# 4. Map Technology Groups for context
tech_map = {
    'SE35': 'BAU (Fossil)',
    'SEFE35': 'Electrified Only', 'SEPH35': 'Electrified Only', 'SEPE35': 'Electrified Only',
    'SEFES35': 'Electrified + Storage', 'SEPHS35': 'Electrified + Storage', 'SEPES35': 'Electrified + Storage'
}
evidence_table['Technology_Group'] = evidence_table['scenario_label'].map(tech_map)

# Final formatting and sorting by cost efficiency
evidence_table = evidence_table.sort_values('EOM_VWAP').round(2)
display(evidence_table)

In [ ]:
import pandas as pd

# 1. Create a temporary copy to avoid modifying the original master_df_eom
temp_df = master_df_eom.copy()

# 2. Safety Check: Ensure 'hour' exists
if 'hour' not in temp_df.columns:
    time_col = [c for c in ['timestamp', 'start_time', 'date'] if c in temp_df.columns]
    if time_col:
        temp_df['hour'] = pd.to_datetime(temp_df[time_col[0]]).dt.hour
    else:
        # If no column found, assume it's in the index
        temp_df['hour'] = temp_df.index.hour

# 3. Find the index of the maximum price for each scenario
# We use idxmax() on the 'accepted_price' column grouped by scenario
idx = temp_df.groupby('scenario_label')['accepted_price'].idxmax()

# 4. Extract the full records for those indices
# We include 'hour' here now that we've guaranteed it exists
highest_price_plants = temp_df.loc[idx, ['scenario_label', 'unit_id', 'accepted_price', 'hour', 'tech_group']]

# 5. Sort for better readability (Highest absolute price at the top)
highest_price_plants = highest_price_plants.sort_values(by='accepted_price', ascending=False)

print("--- Plants Facing Highest EOM Prices per Scenario ---")
display(highest_price_plants)

In [ ]:
# Assuming master_df contains the CRM_pos data
# We combine them and ensure the market_id is clearly labeled
master_df_combined = pd.concat([master_df, master_df_neg], ignore_index=True)

# Clean up numeric types for the new combined dataframe
for col in ['accepted_volume', 'accepted_price']:
    master_df_combined[col] = pd.to_numeric(master_df_combined[col], errors='coerce').fillna(0)

print(f"Combined Data: {len(master_df_combined)} rows total.")

In [ ]:
import matplotlib.pyplot as plt

# 1. Filter for the Storage Scenarios
target_scenarios = ['SEFES35', 'SEPHS35', 'SEPES35']
df_plot = master_df_combined[master_df_combined['scenario_label'].isin(target_scenarios)].copy()

# 2. Calculate Revenue
df_plot['revenue_euro'] = df_plot['accepted_volume'].abs() * df_plot['accepted_price']

# 3. Aggregate for the Stacked Plot
# We group by Scenario, Plant, and market_id (which will be 'CRM_pos' and 'CRM_neg')
plant_agg = df_plot.groupby(['scenario_label', 'unit_id', 'market_id'])['revenue_euro'].sum().unstack().fillna(0)

# 4. Explicit Reorder: CRM_pos at bottom, CRM_neg on top
# Check which columns exist to avoid errors
existing_markets = plant_agg.columns.tolist()
order = [m for m in ['CRM_pos', 'CRM_neg'] if m in existing_markets]
plant_agg = plant_agg[order]
plant_agg_k = plant_agg / 1000  # Scale to k€

# 5. Create Subplots (One per Scenario)
scenarios = plant_agg_k.index.get_level_values('scenario_label').unique()
fig, axes = plt.subplots(len(scenarios), 1, figsize=(14, 6 * len(scenarios)))
if len(scenarios) == 1: axes = [axes]

# Colors: Orange for Pos, Blue for Neg
colors_map = {'CRM_pos': '#e67e22', 'CRM_neg': '#3498db'}
plot_colors = [colors_map.get(c, '#95a5a6') for c in plant_agg_k.columns]

for i, scenario in enumerate(scenarios):
    data_subset = plant_agg_k.loc[scenario].copy()
    
    # Sort by total revenue (most profitable plants on the left)
    data_subset['total'] = data_subset.sum(axis=1)
    data_subset = data_subset.sort_values(by='total', ascending=False).drop(columns='total')
    
    # Plot
    data_subset.plot(kind='bar', stacked=True, ax=axes[i], color=plot_colors, width=0.7, alpha=0.85)
    
    # Formatting
    axes[i].set_title(f'Scenario {scenario}: Total aFRR Revenue per Plant', fontsize=15, fontweight='bold')
    axes[i].set_ylabel('Revenue (k€)', fontsize=12)
    axes[i].set_xlabel('Plant ID (unit_id)', fontsize=12)
    axes[i].legend(title="Market Segment", labels=['Upward (CRM_pos)', 'Downward (CRM_neg)'])
    axes[i].grid(axis='y', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

# 1. Filter and Clean
target_scenarios = ['SEFES35', 'SEPHS35', 'SEPES35']
df_vol = master_df_combined[master_df_combined['scenario_label'].isin(target_scenarios)].copy()

# Force absolute numeric values
df_vol['volume'] = pd.to_numeric(df_vol['volume'], errors='coerce').fillna(0).abs()
df_vol['accepted_volume'] = pd.to_numeric(df_vol['accepted_volume'], errors='coerce').fillna(0).abs()

# 2. Aggregate Volumes (Summed in MWh)
agg_data = df_vol.groupby(['scenario_label', 'market_id'])[['volume', 'accepted_volume']].sum().reset_index()

# 3. Pivot for Stacked Plotting
offered_pivot = agg_data.pivot(index='scenario_label', columns='market_id', values='volume').fillna(0)
accepted_pivot = agg_data.pivot(index='scenario_label', columns='market_id', values='accepted_volume').fillna(0)

# Ensure both CRM_pos and CRM_neg exist in columns
for p in [offered_pivot, accepted_pivot]:
    for col in ['CRM_pos', 'CRM_neg']:
        if col not in p.columns: p[col] = 0
    # Fix column order for consistent stacking
    p = p[['CRM_pos', 'CRM_neg']]

# 4. Create the Figure
fig, ax = plt.subplots(figsize=(14, 8))

n_scenarios = len(target_scenarios)
index = np.arange(n_scenarios)
bar_width = 0.35

# Color Palette: Lighter for 'Offered', Darker for 'Accepted'
colors = {
    'offered_pos': '#ffcc80', 'offered_neg': '#90caf9',
    'accepted_pos': '#e67e22', 'accepted_neg': '#3498db'
}

# --- Plot Group 1: Offered (Stacked) ---
ax.bar(index - bar_width/2, offered_pivot['CRM_pos'], bar_width, 
       label='Offered Downward (Pos)', color=colors['offered_pos'], edgecolor='grey', alpha=0.7)
ax.bar(index - bar_width/2, offered_pivot['CRM_neg'], bar_width, 
       bottom=offered_pivot['CRM_pos'], label='Offered Upward (Neg)', 
       color=colors['offered_neg'], edgecolor='grey', alpha=0.7)

# --- Plot Group 2: Accepted (Stacked) ---
ax.bar(index + bar_width/2, accepted_pivot['CRM_pos'], bar_width, 
       label='Accepted Downward (Pos)', color=colors['accepted_pos'], edgecolor='black')
ax.bar(index + bar_width/2, accepted_pivot['CRM_neg'], bar_width, 
       bottom=accepted_pivot['CRM_pos'], label='Accepted Upward (Neg)', 
       color=colors['accepted_neg'], edgecolor='black')

# 5. FIXING THE Y-AXIS
# Set labels with commas and ensure no scientific notation
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))

# Optional: Conversion to GWh label if numbers are over 10,000
# ax.set_ylabel('Total Volume (GWh)') # If you divide the data by 1000

# 6. Formatting & Labels
plt.title('Cement plant: Offered vs. Accepted Volumes', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Volume (MW)', fontsize=13)
plt.xlabel('Scenario', fontsize=13)
plt.xticks(index, target_scenarios, fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Place legend outside to avoid covering bars
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), frameon=False)

# Add Totals on top of bars
for i in range(n_scenarios):
    off_total = offered_pivot.iloc[i].sum()
    acc_total = accepted_pivot.iloc[i].sum()
    
    ax.text(i - bar_width/2, off_total + (off_total * 0.01), f'{off_total:,.0f}', ha='center', fontsize=10, color='grey')
    ax.text(i + bar_width/2, acc_total + (acc_total * 0.01), f'{acc_total:,.0f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd
import numpy as np

# 1. Clean Numeric Data
master_df_eom['accepted_volume'] = pd.to_numeric(master_df_eom['accepted_volume'], errors='coerce').fillna(0)

# 2. Filter for Target Scenarios
target_scenarios = ['SEFES35', 'SEPHS35', 'SEPES35']
df_eom_plot = master_df_eom[master_df_eom['scenario_label'].isin(target_scenarios)].copy()

# 3. Aggregate Total Volume (Convert to GWh for readability)
# We use .abs() because EOM consumption is typically negative in simulation exports
eom_totals = df_eom_plot.groupby('scenario_label')['accepted_volume'].sum().abs() / 1000

# 4. Visualization
fig, ax = plt.subplots(figsize=(12, 7))

# Create the bars
bars = ax.bar(eom_totals.index, eom_totals.values, color='#8e44ad', alpha=0.8, width=0.6, edgecolor='black')

# 5. FIXING THE Y-AXIS
# Formatter for thousands and standardizing on GWh
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Formatting
plt.title('Total EOM Energy Consumption per Scenario', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Total Accepted Volume (GWh)', fontsize=13)
plt.xlabel('Scenario', fontsize=13)
plt.grid(axis='y', linestyle='--', alpha=0.3)

# Add numeric labels on top of bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + (eom_totals.max() * 0.02),
            f'{height:,.1f} GWh', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# 1. Configuration (Ensuring paths and columns are correct)
# Re-using your scenario paths
scenarios = {
    'SE35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\2\with_storage_trans\market_orders.csv',
    'SEFE35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\4\with_storage_trans\market_orders.csv',
    'SEPH35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\5\with_storage_trans\market_orders.csv',
    'SEPE35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\6\with_storage_trans\market_orders.csv',
    'SEFES35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\10\market_orders.csv',
    'SEPHS35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\11\market_orders.csv',
    'SEPES35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\12\market_orders.csv'
}

cols_to_keep = ['market_id', 'accepted_volume', 'accepted_price']
target_order = ['SE35', 'SEFE35', 'SEPH35', 'SEPE35', 'SEFES35', 'SEPHS35', 'SEPES35']

# 2. System-Wide Extraction
all_data = []

for name, path in scenarios.items():
    print(f"Processing Full Dataset for {name}...")
    try:
        # We read the whole market for CRM_pos and CRM_neg
        for chunk in pd.read_csv(path, usecols=cols_to_keep, chunksize=500000):
            mask = chunk['market_id'].isin(['CRM_pos', 'CRM_neg'])
            filtered_chunk = chunk[mask].copy()
            filtered_chunk['scenario_label'] = name
            all_data.append(filtered_chunk)
    except Exception as e:
        print(f"Could not read {name}: {e}")

df_market = pd.concat(all_data, ignore_index=True)



In [ ]:
# 3. Numeric Cleaning
df_market['accepted_price'] = pd.to_numeric(df_market['accepted_price'], errors='coerce').fillna(0)
df_market['accepted_volume'] = pd.to_numeric(df_market['accepted_volume'], errors='coerce').abs().fillna(0)

# 4. Calculation: Volume-Weighted Average Price (VWAP)
def calculate_vwap(group):
    total_rev = (group['accepted_volume'] * group['accepted_price']).sum()
    total_vol = group['accepted_volume'].sum()
    return total_rev / total_vol if total_vol > 0 else 0

# Group by Scenario and Market, then pivot
market_prices = df_market.groupby(['scenario_label', 'market_id']).apply(calculate_vwap).unstack()

# IMPORTANT: Reindex ensures that SE35, SEPE35, and SEPH35 show up even if their values are 0
market_prices = market_prices.reindex(target_order).fillna(0)

# 5. Plotting
fig, ax = plt.subplots(figsize=(15, 8))
x = np.arange(len(target_order))
width = 0.35

# Plotting the bars
# Use .get() to handle cases where CRM_neg might be entirely missing from the simulation
rects1 = ax.bar(x - width/2, market_prices.get('CRM_pos', 0), width, 
                label='System Avg: pos (Upward)', color='#e67e22', edgecolor='black', alpha=0.9)
rects2 = ax.bar(x + width/2, market_prices.get('CRM_neg', 0), width, 
                label='System Avg: neg (Downward)', color='#3498db', edgecolor='black', alpha=0.9)

# 6. Formatting
ax.set_title('System-Wide Average aFRR Clearing Prices', fontsize=16, fontweight='bold', pad=25)
ax.set_ylabel('Volume-Weighted Average Price (€/MW)', fontsize=13)
ax.set_xlabel('Scenario', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(target_order, fontsize=11)
ax.legend(title="Market Segment", loc='upper right', frameon=True)
ax.grid(axis='y', linestyle='--', alpha=0.3)

# Add value labels on top of bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        if height > 0:
            ax.annotate(f'€{height:.2f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 5), textcoords="offset points",
                        ha='center', va='bottom', fontsize=10, fontweight='bold')
        else:
            ax.annotate('€0.00', xy=(rect.get_x() + rect.get_width() / 2, 0),
                        xytext=(0, 5), textcoords="offset points",
                        ha='center', va='bottom', fontsize=9, color='gray')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Full EOM Data Extraction
all_eom_data = []
target_order = ['SE35', 'SEFE35', 'SEPH35', 'SEPE35', 'SEFES35', 'SEPHS35', 'SEPES35']

for name, path in scenarios_eom.items():
    print(f"Processing Full EOM Dataset for {name}...")
    # Use cols_to_keep from your previous setup (market_id, accepted_price, etc.)
    for chunk in pd.read_csv(path, usecols=cols_to_keep, chunksize=500000):
        mask = (chunk['market_id'] == 'EOM')
        filtered_chunk = chunk[mask].copy()
        filtered_chunk['scenario_label'] = name
        all_eom_data.append(filtered_chunk)

df_eom_full = pd.concat(all_eom_data, ignore_index=True)

# 2. Clean Numeric Data
df_eom_full['accepted_price'] = pd.to_numeric(df_eom_full['accepted_price'], errors='coerce')



In [ ]:
# 3. Calculate Stats: Min, Max, and Average
eom_stats = df_eom_full.groupby('scenario_label')['accepted_price'].agg(['min', 'max', 'mean']).reindex(target_order)

# 4. Plotting
fig, ax = plt.subplots(figsize=(15, 8))

x = np.arange(len(target_order))
width = 0.25

# Create bars for Min, Avg, and Max
ax.bar(x - width, eom_stats['min'], width, label='Min Price', color='#2ecc71', alpha=0.7)
ax.bar(x, eom_stats['mean'], width, label='Avg Price (Mean)', color='#f1c40f', alpha=0.9)
ax.bar(x + width, eom_stats['max'], width, label='Max Price', color='#e74c3c', alpha=0.7)

# 5. Formatting
ax.set_title('DayAhead Market Price Range: Min, Max, and Average per Scenario', fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('Market Price (€/MWh)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(target_order, fontsize=11)
ax.legend(loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.3)

# Add text labels for the Mean values
for i, val in enumerate(eom_stats['mean']):
    ax.text(i, val + (eom_stats['max'].max() * 0.02), f'€{val:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# 6. Display Numerical Table
print("\n--- DayAhead Price Statistics Table (€/MWh) ---")
print(eom_stats.round(2))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Configuration: Update paths if necessary
scenarios = {
    'SE35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\2\with_storage_trans\market_orders.csv',
    'SEFE35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\4\with_storage_trans\market_orders.csv',
    'SEPH35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\5\with_storage_trans\market_orders.csv',
    'SEPE35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\6\with_storage_trans\market_orders.csv',
    'SEFES35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\10\market_orders.csv',
    'SEPHS35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\11\market_orders.csv',
    'SEPES35': r'C:\Users\khm\ownCloud (2)\Dropbox\Ph.D\My publications\Journal paper\3\Simulation\Output\12\market_orders.csv'
}

# Column name check: 'offered_volume' is the standard in most CRM simulation outputs
cols_to_extract = ['market_id', 'volume', 'accepted_volume']
target_order = ['SE35', 'SEFE35', 'SEPH35', 'SEPE35', 'SEFES35', 'SEPHS35', 'SEPES35']

# 2. Extraction Loop
volume_data = []

for name, path in scenarios.items():
    print(f"Aggregating system volumes for {name}...")
    try:
        t_off_pos, t_off_neg = 0, 0
        t_acc_pos, t_acc_neg = 0, 0
        
        for chunk in pd.read_csv(path, usecols=cols_to_extract, chunksize=500000):
            # Clean and take absolute values to handle negative sign conventions
            chunk['volume'] = pd.to_numeric(chunk['volume'], errors='coerce').abs().fillna(0)
            chunk['accepted_volume'] = pd.to_numeric(chunk['accepted_volume'], errors='coerce').abs().fillna(0)
            
            # Filter by market type
            pos_mask = (chunk['market_id'] == 'CRM_pos')
            neg_mask = (chunk['market_id'] == 'CRM_neg')
            
            t_off_pos += chunk[pos_mask]['volume'].sum()
            t_acc_pos += chunk[pos_mask]['accepted_volume'].sum()
            
            t_off_neg += chunk[neg_mask]['volume'].sum()
            t_acc_neg += chunk[neg_mask]['accepted_volume'].sum()
            
        volume_data.append({
            'Scenario': name,
            'Offered_Pos': t_off_pos,
            'Offered_Neg': t_off_neg,
            'Accepted_Pos': t_acc_pos,
            'Accepted_Neg': t_acc_neg
        })
    except Exception as e:
        print(f"Skip {name}: {e}")

df_vol_summary = pd.DataFrame(volume_data).set_index('Scenario').reindex(target_order).fillna(0)


In [ ]:
import matplotlib.ticker as ticker

# 3. Plotting: Grouped and Stacked Bar Chart
fig, ax = plt.subplots(figsize=(15, 8))
x = np.arange(len(target_order))
width = 0.4

# Bars for Offered (Left side of the group)
ax.bar(x - width/2, df_vol_summary['Offered_Neg'], width, label='Offered: Negative', color='#aed6f1', edgecolor='black')
ax.bar(x - width/2, df_vol_summary['Offered_Pos'], width, bottom=df_vol_summary['Offered_Neg'], 
       label='Offered: Positive', color='#f9e79f', edgecolor='black')

# Bars for Accepted (Right side of the group)
ax.bar(x + width/2, df_vol_summary['Accepted_Neg'], width, label='Accepted: Negative', color='#3498db', edgecolor='black')
ax.bar(x + width/2, df_vol_summary['Accepted_Pos'], width, bottom=df_vol_summary['Accepted_Neg'], 
       label='Accepted: Positive', color='#f39c12', edgecolor='black')

# --- Y-AXIS FIXES ---
# Format Y-axis with commas for readability (e.g., 1,000,000)
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Ensure Y-axis starts exactly at 0 and gives 15% headroom at the top for labels
max_val = max(df_vol_summary['Offered_Pos'] + df_vol_summary['Offered_Neg'])
ax.set_ylim(0, max_val * 1.15) 

# 4. Formatting
ax.set_title('System-Wide aFRR Volumes: Offered vs. Accepted', fontsize=16, fontweight='bold', pad=20)
ax.set_ylabel('Total Annual Volume (MW)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(target_order, fontsize=11)

# Move legend inside or adjust to ensure it doesn't overlap
ax.legend(title="Volume Category", loc='upper right', frameon=True, shadow=True)
ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)

# Set bars to be on top of grid
ax.set_axisbelow(True)

# Label the X-axis sub-groups ('Offered' vs 'Accepted') for the first scenario as an example
ax.text(x[0]-width/2, -max_val*0.05, 'Offered', ha='center', fontsize=10, fontweight='bold', color='gray')
ax.text(x[0]+width/2, -max_val*0.05, 'Accepted', ha='center', fontsize=10, fontweight='bold', color='gray')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 1. Prepare Data: Calculate Weekly Price (€/MW/Week)
# Average Hourly Price * 168 hours in a week
weekly_prices = df_market.groupby(['scenario_label', 'market_id'])['accepted_price'].mean().unstack()
weekly_prices_eur_week = weekly_prices * 168

# 2. Reshape for Seaborn (Long Format)
# This turns the columns 'CRM_pos' and 'CRM_neg' into a single 'Direction' column
df_long = weekly_prices_eur_week.reset_index().melt(
    id_vars='scenario_label', 
    var_name='Market_Direction', 
    value_name='Price_Weekly'
)

# Rename for better legend labels
df_long['Market_Direction'] = df_long['Market_Direction'].replace({
    'CRM_pos': 'Positive (Upward)', 
    'CRM_neg': 'Negative (Downward)'
})

# 3. Plotting with Seaborn
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 8))

# Creating the grouped bar plot
# dodge=True ensures the bars are side-by-side but distinct
ax = sns.barplot(
    data=df_long, 
    x='scenario_label', 
    y='Price_Weekly', 
    hue='Market_Direction',
    order=target_order,
    palette=['#FFA07A', '#5DADE2'], # Distinct colors for Up vs Down
    edgecolor='black'
)

# 4. Refined Formatting
plt.title('Market Capacity Price: Positive vs. Negative Comparison', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Scenario', fontsize=12, fontweight='bold')
plt.ylabel('Price (€ / MW / Week)', fontsize=12, fontweight='bold')

# Fix Y-axis: Add commas and ensure it starts at 0
ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Add data labels on top of each bar
for container in ax.containers:
    ax.bar_label(container, fmt='{:,.0f}', padding=3, fontweight='bold', fontsize=10)

# Adjust legend and grid
plt.legend(title='aFRR Direction', loc='upper right', frameon=True)
sns.despine(offset=10, trim=True) # Makes the plot look "cleaner" and academic

# Headroom for labels
plt.ylim(0, df_long['Price_Weekly'].max() * 1.15)

plt.tight_layout()
plt.show()

In [ ]:
# Assuming your df_market is 8760 hours long per scenario
# We can create a quarter tag (Approximate: 2190 hours per quarter)
df_market['hour_index'] = df_market.index % 8760 
df_market['Quarter'] = pd.cut(df_market['hour_index'], 
                              bins=[0, 2190, 4380, 6570, 8760], 
                              labels=['Q1', 'Q2', 'Q3', 'Q4'])

# Now we can plot the Average Weekly Price PER QUARTER
quarterly_stats = df_market.groupby(['scenario_label', 'Quarter', 'market_id'])['accepted_price'].mean() * 168
quarterly_stats = quarterly_stats.unstack().reset_index()

# Plotting SEFE35 vs SEFES35 by Quarter
plt.figure(figsize=(12, 6))
sns.barplot(data=quarterly_stats[quarterly_stats['scenario_label'].isin(['SEFE35', 'SEFES35'])], 
            x='Quarter', y='CRM_neg', hue='scenario_label')

plt.title('Seasonal Investigation: Weekly Price by Quarter', fontsize=14)
plt.ylabel('€ / MW / Week')
plt.show()

In [ ]:
# 1. Manually recreate the hour column based on row position
# This assumes each scenario has a full set of sequential hours
df_market['hour'] = df_market.groupby('scenario_label').cumcount()

# 2. Define the Quarters based on that hour
def get_quarter(h):
    if h < 2190: return 'Q1 (Winter)'   # Jan - Mar
    elif h < 4380: return 'Q2 (Spring)' # Apr - Jun
    elif h < 6570: return 'Q3 (Summer)' # Jul - Sep
    else: return 'Q4 (Autumn)'          # Oct - Dec

df_market['Quarter'] = df_market['hour'].apply(get_quarter)

# 3. Calculate Weekly Price (€/MW/Week) 
quarterly_data = df_market.groupby(['scenario_label', 'Quarter', 'market_id'])['accepted_price'].mean().reset_index()
quarterly_data['Price_Weekly'] = quarterly_data['accepted_price'] * 168

# 4. Plotting with Seaborn
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

sns.set_theme(style="whitegrid")
target_order = ['SE35', 'SEFE35', 'SEPH35', 'SEPE35', 'SEFES35', 'SEPHS35', 'SEPES35']

g = sns.catplot(
    data=quarterly_data, 
    kind="bar",
    x="Quarter", 
    y="Price_Weekly", 
    hue="scenario_label",
    col="market_id", 
    hue_order=target_order,
    palette="viridis",
    height=6, 
    aspect=1.2,
    edgecolor='black'
)

# Formatting
g.set_axis_labels("Season", "Price (€ / MW / Week)", fontweight='bold')
g.set_titles("{col_name}", fontweight='bold')
for ax in g.axes.flat:
    ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

plt.subplots_adjust(top=0.85)
g.fig.suptitle('Quarterly Market Price Comparison across Scenarios', fontsize=16, fontweight='bold')

plt.show()